In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ctrl-is-hers-final/sample_submission.csv
/kaggle/input/competitions/ctrl-is-hers-final/train.csv
/kaggle/input/competitions/ctrl-is-hers-final/metaData.csv
/kaggle/input/competitions/ctrl-is-hers-final/test.csv


In [2]:
train= pd.read_csv("/kaggle/input/competitions/ctrl-is-hers-final/train.csv")
test=pd.read_csv("/kaggle/input/competitions/ctrl-is-hers-final/test.csv")
submission_sample= pd.read_csv("/kaggle/input/competitions/ctrl-is-hers-final/sample_submission.csv")
print(train.columns.tolist())

['id', 'publish_date', 'weekday', 'channel', 'n_tokens_title', 'n_tokens_content', 'n_unique_tokens', 'n_non_stop_words', 'n_non_stop_unique_tokens', 'num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos', 'average_token_length', 'num_keywords', 'kw_min_min', 'kw_max_min', 'kw_avg_min', 'kw_min_max', 'kw_max_max', 'kw_avg_max', 'kw_min_avg', 'kw_max_avg', 'kw_avg_avg', 'self_reference_min_shares', 'self_reference_max_shares', 'self_reference_avg_shares', 'LDA_00', 'LDA_01', 'LDA_02', 'LDA_03', 'LDA_04', 'global_subjectivity', 'global_sentiment_polarity', 'global_rate_positive_words', 'global_rate_negative_words', 'rate_positive_words', 'rate_negative_words', 'avg_positive_polarity', 'min_positive_polarity', 'max_positive_polarity', 'avg_negative_polarity', 'min_negative_polarity', 'max_negative_polarity', 'title_subjectivity', 'title_sentiment_polarity', 'abs_title_subjectivity', 'abs_title_sentiment_polarity', 'target']


In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31715 entries, 0 to 31714
Data columns (total 49 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            31715 non-null  int64  
 1   publish_date                  31715 non-null  object 
 2   weekday                       31715 non-null  object 
 3   channel                       31715 non-null  object 
 4   n_tokens_title                31715 non-null  float64
 5   n_tokens_content              31715 non-null  float64
 6   n_unique_tokens               31715 non-null  float64
 7   n_non_stop_words              31715 non-null  float64
 8   n_non_stop_unique_tokens      31715 non-null  float64
 9   num_hrefs                     31715 non-null  float64
 10  num_self_hrefs                31715 non-null  float64
 11  num_imgs                      31715 non-null  float64
 12  num_videos                    31715 non-null  float64
 13  a

In [4]:
train.isnull().sum()

id                              0
publish_date                    0
weekday                         0
channel                         0
n_tokens_title                  0
n_tokens_content                0
n_unique_tokens                 0
n_non_stop_words                0
n_non_stop_unique_tokens        0
num_hrefs                       0
num_self_hrefs                  0
num_imgs                        0
num_videos                      0
average_token_length            0
num_keywords                    0
kw_min_min                      0
kw_max_min                      0
kw_avg_min                      0
kw_min_max                      0
kw_max_max                      0
kw_avg_max                      0
kw_min_avg                      0
kw_max_avg                      0
kw_avg_avg                      0
self_reference_min_shares       0
self_reference_max_shares       0
self_reference_avg_shares       0
LDA_00                          0
LDA_01                          0
LDA_02        

In [5]:
train["target"].value_counts()

target
1    17357
0    14358
Name: count, dtype: int64

In [6]:
train["publish_date"]=pd.to_datetime(train["publish_date"])
test["publish_date"]=pd.to_datetime(test["publish_date"])

In [7]:
train["month"]= train["publish_date"].dt.month
train["day"]= train["publish_date"].dt.day
train["month"]= test["publish_date"].dt.month
train["day"]= test["publish_date"].dt.day


In [8]:
train.drop(columns=["publish_date"], inplace=True)
test.drop(columns=["publish_date"], inplace=True)


In [9]:
X=train.drop(columns=["target", "id"])
y=train["target"]
X_test= test.drop(columns=["id"])

In [10]:
num_col=X.select_dtypes(
    include=["int64", "float64"]
).columns

cat_col=X.select_dtypes(
    include=["object"]
).columns

In [11]:
print(list(num_col))
print(list(cat_col))

['n_tokens_title', 'n_tokens_content', 'n_unique_tokens', 'n_non_stop_words', 'n_non_stop_unique_tokens', 'num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos', 'average_token_length', 'num_keywords', 'kw_min_min', 'kw_max_min', 'kw_avg_min', 'kw_min_max', 'kw_max_max', 'kw_avg_max', 'kw_min_avg', 'kw_max_avg', 'kw_avg_avg', 'self_reference_min_shares', 'self_reference_max_shares', 'self_reference_avg_shares', 'LDA_00', 'LDA_01', 'LDA_02', 'LDA_03', 'LDA_04', 'global_subjectivity', 'global_sentiment_polarity', 'global_rate_positive_words', 'global_rate_negative_words', 'rate_positive_words', 'rate_negative_words', 'avg_positive_polarity', 'min_positive_polarity', 'max_positive_polarity', 'avg_negative_polarity', 'min_negative_polarity', 'max_negative_polarity', 'title_subjectivity', 'title_sentiment_polarity', 'abs_title_subjectivity', 'abs_title_sentiment_polarity', 'month', 'day']
['weekday', 'channel']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [ ]:
num_transfor= Pipeline ([
    ("imputer", SimpleImputer(strategy="median"))
])

In [ ]:
cat_transfor= Pipeline ([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [ ]:
preprocessor = ColumnTransformer([
    ("num", num_transfor,num_col),
    ("cat", cat_transfor,cat_col)
])

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model1= Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

In [ ]:
model1.fit(X_train, y_train)

In [ ]:
pred1= model1.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
print("LogReg", accuracy_score(y_val, pred1))

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

In [ ]:
model2= Pipeline([
    ("preprocessor",preprocessor),

    ("model", ExtraTreesClassifier(
        n_estimators=300,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ) )
])

In [ ]:
model2.fit(X_train, y_train)

In [ ]:
pred2= model2.predict(X_val)

print(
    "ExtraTreesClassifier acc",
    accuracy_score(y_val, pred2)
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model3 = Pipeline([
    ("preprocessor", preprocessor),

    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=15,
        min_samples_split=4,
        random_state=42,
        n_jobs=-1
    ))
])



In [ ]:
model3.fit(X_train, y_train)

pred3 = model3.predict(X_val)

print(
    "Random Forest acc",
    accuracy_score(y_val, pred3)
)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

model4 = Pipeline([
    ("preprocessor", preprocessor),

    ("model", GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])


In [ ]:
model4.fit(X_train, y_train)

In [ ]:
pred4 = model4.predict(X_val)
print(
    "Gradient Boosting acc",
    accuracy_score(y_val, pred4)
)

In [ ]:
from sklearn.metrics import roc_auc_score

prob1= model1.predict_proba(X_val)[:,1]
prob2= model2.predict_proba(X_val)[:,1]
prob3= model3.predict_proba(X_val)[:,1]
prob4= model4.predict_proba(X_val)[:,1]

auc1= roc_auc_score(y_val, prob1)
auc2= roc_auc_score(y_val, prob2)
auc3= roc_auc_score(y_val, prob3)
auc4= roc_auc_score(y_val, prob4)

print( "log", auc1)
print( "extra", auc2)
print( "random", auc3)
print( "GB", auc4)

In [ ]:
model4.fit(X,y)

In [ ]:
test["publish_date"] = pd.to_datetime(test["publish_date"])

test["month"] = test["publish_date"].dt.month
test["day"] = test["publish_date"].dt.day

In [ ]:
print(set(X.columns) - set(X_test.columns))

In [ ]:
X_test = test.drop(columns=["id", "publish_date"])

In [ ]:
print(set(X.columns) - set(X_test.columns))

In [ ]:
model4.fit(X, y)

In [ ]:
test_pred = model4.predict_proba(X_test)[:, 1]

In [ ]:
submission = submission_sample.copy()
submission["target"] = test_pred

submission.to_csv(
    "/kaggle/working/submission_gradient.csv",
    index=False
)

submission.head()